In [ ]:
import pandas as pd
import numpy as np

import os
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as st

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import coint

import warnings

warnings.filterwarnings("ignore")

os.makedirs('images', exist_ok=True)

In [ ]:
df = pd.read_csv("marketing_campaign_large.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df['Date'].dtype

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(15,6))

plt.subplot(1,2,1)
plt.title('Facebook Ad Clicks')
sns.histplot(df['Facebook_Ad_Clicks'], bins=7, edgecolor='k', kde=True)

plt.subplot(1,2,2)
plt.title('Facebook Ad Conversions')
sns.histplot(df['Facebook_Ad_Conversions'], bins=7, edgecolor='k', kde=True)

plt.savefig('images/facebook_ad_histograms.png', bbox_inches='tight')
plt.show()


plt.figure(figsize=(15,6))

plt.subplot(1,2,1)
plt.title('AdWords Ad Clicks')
sns.histplot(df['AdWords_Ad_Clicks'], bins=7, edgecolor='k', kde=True)

plt.subplot(1,2,2)
plt.title('AdWords Ad Conversions')
sns.histplot(df['AdWords_Ad_Conversions'], bins=7, edgecolor='k', kde=True)

plt.savefig('images/adwords_ad_histograms.png', bbox_inches='tight')
plt.show()

In [ ]:
def create_conversion_category(conversion_col):
    category = []

    for conversion in df[conversion_col]:
        if conversion < 20:
            category.append('less than 20')

        elif 20 <= conversion < 40:
            category.append('20 - 39')

        elif 40 <= conversion < 60:
            category.append('40 - 59')

        else:
            category.append('more than 60')

    return category

df['Facebook_Conversion_Category'] = create_conversion_category('Facebook_Ad_Conversions')

df['AdWords_Conversion_Category'] = create_conversion_category('AdWords_Ad_Conversions')

In [ ]:
df[['Facebook_Ad_Conversions',
    'Facebook_Conversion_Category',
    'AdWords_Ad_Conversions',
    'AdWords_Conversion_Category']].head()

In [ ]:
df['Facebook_Conversion_Category'].value_counts()

In [ ]:
facebook = (
    pd.DataFrame(df['Facebook_Conversion_Category'].value_counts())
    .reset_index()
    .rename(columns={
        'Facebook_Conversion_Category': 'Count',
        'index': 'Category'
    })
)

facebook

In [ ]:
df['AdWords_Conversion_Category'].value_counts()

In [ ]:
adwords = (
    pd.DataFrame(df['AdWords_Conversion_Category'].value_counts())
    .reset_index()
    .rename(columns={
        'AdWords_Conversion_Category': 'Count',
        'index': 'Category'
    })
)

adwords

In [ ]:
facebook = df['Facebook_Conversion_Category'].value_counts().reset_index()
facebook.columns = ['Category', 'Facebook_Count']

adwords = df['AdWords_Conversion_Category'].value_counts().reset_index()
adwords.columns = ['Category', 'AdWords_Count']

In [ ]:
category_df = pd.merge(
    facebook,
    adwords,
    on='Category',
    how='outer'
).fillna(0)

category_df

In [ ]:
desired_order = ['less than 20', '20 - 39', '40 - 59', 'more than 60']

category_df = (
    category_df
    .set_index('Category')
    .reindex(desired_order, fill_value=0)
    .reset_index()
)

category_df

In [ ]:
X_axis = np.arange(len(category_df))

plt.figure(figsize=(15,6))

plt.bar(
    X_axis - 0.2,
    category_df['Facebook_Count'],
    0.4,
    label='Facebook',
    edgecolor='k'
)

plt.bar(
    X_axis + 0.2,
    category_df['AdWords_Count'],
    0.4,
    label='AdWords',
    edgecolor='k'
)

plt.xticks(X_axis, category_df['Category'])
plt.xlabel("Conversion Category")
plt.ylabel("Number of Days")
plt.title("Frequency of Daily Conversions by Conversion Categories")
plt.legend()
plt.savefig('images/conversion_category_frequency.png', bbox_inches='tight')

plt.show()

In [ ]:
plt.figure(figsize=(15,6))

plt.subplot(1,2,1)
plt.title('Facebook')
sns.scatterplot(
    x=df['Facebook_Ad_Clicks'],
    y=df['Facebook_Ad_Conversions']
)
plt.xlabel('Clicks')
plt.ylabel('Conversions')

plt.subplot(1,2,2)
plt.title('AdWords')
sns.scatterplot(
    x=df['AdWords_Ad_Clicks'],
    y=df['AdWords_Ad_Conversions']
)
plt.xlabel('Clicks')
plt.ylabel('Conversions')

plt.savefig('images/clicks_vs_conversions_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
facebook_corr = df[['Facebook_Ad_Conversions', 'Facebook_Ad_Clicks']].corr()

facebook_corr

In [ ]:
adwords_corr = df[['AdWords_Ad_Conversions', 'AdWords_Ad_Clicks']].corr()

adwords_corr

In [ ]:
print('Correlation Coeff\n----------------')

print('Facebook :', round(facebook_corr.values[0,1], 2))
print('AdWords  :', round(adwords_corr.values[0,1], 2))

In [ ]:
print('Mean Conversion\n----------------')

print('Facebook :', round(df['Facebook_Ad_Conversions'].mean(), 2))
print('AdWords  :', round(df['AdWords_Ad_Conversions'].mean(), 2))


t_stats, p_value = st.ttest_ind(
    a=df['Facebook_Ad_Conversions'],
    b=df['AdWords_Ad_Conversions'],
    equal_var=False
)

print('\nT statistic:', t_stats)
print('P-value:', p_value)


# comparing the p-value with significance level of 0.05

if p_value < 0.05:
    print('\nP-value is less than significance value, Reject the null hypothesis')
else:
    print('\nP-value is greater than significance value, Accept the null hypothesis')

In [ ]:
# independent variable

X = df[['Facebook_Ad_Clicks']]

# dependent variable

y = df[['Facebook_Ad_Conversions']]

# initializing and fitting Linear Regression model

reg_model = LinearRegression()

reg_model.fit(X, y)

prediction = reg_model.predict(X)

# model evaluation

r2 = r2_score(y, prediction) * 100

mse = mean_squared_error(y, prediction)

print('Accuracy (R2 Score):', round(r2, 2), '%')
print('Mean Squared Error:', round(mse, 2))

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    x=df['Facebook_Ad_Clicks'],
    y=df['Facebook_Ad_Conversions'],
    label='Actual data points'
)

plt.plot(
    df['Facebook_Ad_Clicks'],
    prediction,
    label='Best fit line'
)

plt.legend()
plt.savefig('images/facebook_clicks_vs_conversions_regression.png', bbox_inches='tight')

plt.show()

In [ ]:
print(f'For 50 Clicks, Expected Conversion : {round(reg_model.predict([[50]])[0][0], 2)}')

print(f'For 80 Clicks, Expected Conversion : {round(reg_model.predict([[80]])[0][0], 2)}')

In [ ]:
# extracting month and week day from the date column

df['month'] = df['Date'].dt.month
df['week'] = df['Date'].dt.weekday

In [ ]:
plt.figure(figsize=(8,5))

plt.title('Weekly Conversions')

weekly_conversion = df.groupby('week')[['Facebook_Ad_Conversions']].sum()

week_names = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

plt.bar(
    week_names,
    weekly_conversion['Facebook_Ad_Conversions'],
    edgecolor='k'
)

plt.savefig('images/weekly_conversions.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.title('Monthly Conversions')

monthly_conversion = df.groupby('month')[['Facebook_Ad_Conversions']].sum()

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.plot(
    month_names,
    monthly_conversion['Facebook_Ad_Conversions'],
    '-o'
)

plt.savefig('images/monthly_conversions.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.title('Monthly Cost Per Conversion (CPC)')

monthly_df = df.groupby('month')[['Facebook_Ad_Conversions', 'Cost_Per_Facebook_Ad']].sum()

monthly_df['Cost_Per_Conversion'] = (
    monthly_df['Cost_Per_Facebook_Ad'] /
    monthly_df['Facebook_Ad_Conversions']
)

plt.plot(
    month_names,
    monthly_df['Cost_Per_Conversion'],
    '-o'
)

plt.savefig('images/monthly_cost_per_conversion.png', bbox_inches='tight')
plt.show()

In [ ]:
score, p_value, _ = coint(
    df['Cost_Per_Facebook_Ad'],
    df['Facebook_Ad_Conversions']
)

print('Cointegration test score:', score)
print('P-value:', p_value)

if p_value < 0.05:
    print('\nP-value is less than significance value, Reject the null hypothesis')
else:
    print('\nP-value is greater than significance value, Accept the null hypothesis')